# (27) animation — mnist to imgnet (save mp4)

**Motivation**: host = ```mach```, device = ```cuda:1``` <br>

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-vae/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-vae/figs')
tmp_dir = os.path.join(git_dir, 'jb-vae/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_IterativeVAE/iclr_submitted'))
from figures.fighelper import *
from vae.train_vae import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

In [2]:
from analysis.chosen import *
from figures.analysis import *
from analysis.final import save_both_row_col

from figures.animate import plot_frame

device_idx = 1
device = f'cuda:{device_idx}'

print(f"device: {device}  ———  host: {os.uname().nodename}")

device: cuda:1  ———  host: mach

In [3]:
anim_dir = pjoin(fig_base_dir, 'animation')
os.makedirs(anim_dir, exist_ok=True)
print(os.listdir(anim_dir))

[
    'material',
    'mnist-to-imgnet.mp4',
    'output_video.mp4',
    'mnist-to-imgnet_dpi-140.mp4',
    'output-video_zAAT2Mid.mp4',
    'mnist-to-imgnet_dpi-300.mp4',
    'mnist-to-imgnet_dpi-150.mp4'
]

In [4]:
from moviepy.editor import ImageClip, concatenate_videoclips

import moviepy
moviepy.__version__

'1.0.3'

In [5]:
dpi = 300
load_dir = pjoin(anim_dir, 'material', f"mnist-to-imgnet_dpi-{dpi}")
filename_pattern = r'mnist-to-imgnet_t=(\d+)\.png'

image_files = []
for f in sorted(os.listdir(load_dir), key=alphanum_sort_key):
    match = re.match(filename_pattern, f)
    if match:
        t = int(match.group(1))
        image_files.append((t, os.path.join(load_dir, f)))
image_files.sort(key=lambda x: x[0])

In [6]:
sorted_image_paths = [file_path for _, file_path in image_files]
n_frames = len(sorted_image_paths)
n_frames

1000

In [7]:
clips = []

for idx, image_path in tqdm(enumerate(sorted_image_paths), total=n_frames):
    if (idx < 100) or (idx in 100+np.cumsum(np.arange(1, 500, 1))):
        clip = ImageClip(image_path).set_duration(1/18)
        clips.append(clip)

100%|██████████████████████████████████████| 1000/1000 [00:09<00:00, 109.55it/s]


In [8]:
pause_start_duration = 0.5   # pause on the first frame (seconds)
pause_end_duration = 2.5     # pause last frame (seconds)

start_pause_clip = ImageClip(sorted_image_paths[0]).set_duration(pause_start_duration)
end_pause_clip = ImageClip(sorted_image_paths[-1]).set_duration(pause_end_duration)

video = concatenate_videoclips([start_pause_clip] + clips + [end_pause_clip], method="compose")

In [9]:
%%time


vid_file = f'mnist-to-imgnet_dpi-{dpi}.mp4'
vid_file = pjoin(anim_dir, vid_file)

video.write_videofile(
    filename=vid_file,
    fps=60,                 # Frames per second (adjust as needed)
    codec='libx264',        # Video codec
    bitrate='5000k',        # Bitrate for higher quality
    audio=False,            # No audio
    threads=16,             # Number of threads for encoding
    preset='medium',        # Encoding speed/quality trade-off
)

Moviepy - Building video /home/hadi/Dropbox/git/jb-vae/figs/animation/mnist-to-imgnet_dpi-300.mp4.
Moviepy - Writing video /home/hadi/Dropbox/git/jb-vae/figs/animation/mnist-to-imgnet_dpi-300.mp4



Moviepy - Done !
Moviepy - video ready /home/hadi/Dropbox/git/jb-vae/figs/animation/mnist-to-imgnet_dpi-300.mp4
CPU times: user 1min 9s, sys: 36.2 s, total: 1min 45s
Wall time: 1min 48s


## Resize?

TODO

## Test if it's working

In [5]:
import moviepy.config as conf
import moviepy.tools as tools

In [6]:
print(conf.get_setting("FFMPEG_BINARY"))

/home/hadi/anaconda3/bin/ffmpeg

In [8]:
from moviepy.config import FFMPEG_BINARY
import os

def check_ffmpeg():
    try:
        # Check if the FFMPEG binary exists
        if os.path.exists(FFMPEG_BINARY):
            print(f"FFMPEG found at: {FFMPEG_BINARY}")
            
            # Try to run ffmpeg to verify it works
            result = os.system(f"{FFMPEG_BINARY} -version")
            if result == 0:
                print("FFMPEG is working correctly!")
            else:
                print("FFMPEG found but not working properly")
        else:
            print(f"FFMPEG not found at: {FFMPEG_BINARY}")
            
    except Exception as e:
        print(f"Error checking FFMPEG: {e}")

# Run the check
check_ffmpeg()

FFMPEG found at: /home/hadi/anaconda3/bin/ffmpeg

ffmpeg version 6.0 Copyright (c) 2000-2023 the FFmpeg developers
built with gcc 12.2.0 (conda-forge gcc 12.2.0-19)
configuration: --prefix=/home/hadi/anaconda3 --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1684241469673/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1684241469673/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=/home/conda/feedstock_root/build_artifacts/ffmpeg_1684241469673/_build_env/bin/x86_64-conda-linux-gnu-nm --ar=/home/conda/feedstock_root/build_artifacts/ffmpeg_1684241469673/_build_env/bin/x86_64-conda-linux-gnu-ar --disable-doc --disable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-gnutls --enable-libmp3lame --enable-libvpx --enable-libass --enable-pthreads --enable-vaapi --enable-gpl --enable-libx264 --enable-libx265 --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disa

FFMPEG is working correctly!